# IV_07 — Anomalías y vida útil remanente (RUL)

## 1. Objetivo

Detectar anomalías en vibración y estimar RUL (Remaining Useful Life) para programar intervenciones antes de la falla funcional.

## 2. Concepto

- **Umbral fijo (ISO 10816):** simple pero no detecta degradación gradual.
- **Z-score:** alerta si valor > μ + 2σ.
- **Isolation Forest:** detecta patrones multivariados anómalos.
- **RUL:** horas estimadas hasta intervención recomendada.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

MOD_DIR = Path.cwd()
os.chdir(MOD_DIR)
DATA_DIR = MOD_DIR / "data"
OUTPUT_DIR = MOD_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
DATA_PATH = DATA_DIR / "dataset_predictivo.csv"

from sklearn.ensemble import GradientBoostingRegressor, IsolationForest
from sklearn.model_selection import train_test_split


In [ ]:
df = pd.read_csv(DATA_PATH, parse_dates=["Timestamp"])
vib = df["PUMP101.VIBRATION_RMS"]

# Detección por z-score
media, std = vib.mean(), vib.std()
umbral_z = media + 2 * std
anomalias_z = vib > umbral_z
print(f"Umbral z-score: {umbral_z:.2f} mm/s | Anomalías: {anomalias_z.sum()}")


In [ ]:
# Isolation Forest multivariable
feat_cols = ["PUMP101.VIBRATION_RMS", "PUMP101.BEARING_TEMP", "vib_media_24h"]
iso = IsolationForest(contamination=0.05, random_state=42)
df["anomalia"] = iso.fit_predict(df[feat_cols])
df["anomalia_label"] = df["anomalia"].map({1: "Normal", -1: "Anomalía"})
print(df["anomalia_label"].value_counts())


In [ ]:
# RUL simplificado: regresión sobre índice temporal
df["horas_operacion"] = np.arange(len(df))
rul_target = np.clip(500 - df["PUMP101.VIBRATION_RMS"] * 80 - df["PUMP101.BEARING_TEMP"] * 2, 10, 500)

X = df[["PUMP101.VIBRATION_RMS", "PUMP101.BEARING_TEMP", "vib_media_24h", "horas_operacion"]]
y = rul_target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rul_model = GradientBoostingRegressor(random_state=42)
rul_model.fit(X_train, y_train)
rul_pred = rul_model.predict(X_test)

rul_actual = float(rul_model.predict(X.iloc[[-1]])[0])
print(f"RUL estimado actual: {rul_actual:.0f} horas")


## 3. Visualización

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

axes[0].plot(df["Timestamp"], vib, alpha=0.6, label="Vibración RMS")
axes[0].axhline(umbral_z, color="red", linestyle="--", label="Umbral z-score")
anom_idx = df[df["anomalia"] == -1]
axes[0].scatter(anom_idx["Timestamp"], anom_idx["PUMP101.VIBRATION_RMS"], color="red", s=20, label="Anomalía")
axes[0].set_ylabel("mm/s")
axes[0].set_title("Detección de anomalías — PUMP101")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].scatter(y_test, rul_pred, alpha=0.5)
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--")
axes[1].set_xlabel("RUL real (h)")
axes[1].set_ylabel("RUL predicho (h)")
axes[1].set_title("Modelo RUL")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "anomalias_rul.png", dpi=150)


## 4. Interpretación para mantenimiento

Combinar umbral ISO + detección estadística reduce falsas alarmas. RUL < 72 h sugiere programar inspección predictiva en la próxima ventana de parada.

## 5. Resumen y siguiente paso

- Anomalías multivariables captan degradación que un solo umbral no ve.
- RUL traduce sensores en tiempo hasta intervención.
- Conectar con Lab 06 (vibración) para umbrales ISO.

**Siguiente:** `IV_08_caso_integrado_alertas.ipynb`